# Adaptive Residual Steering (ARS) — Kaggle Execution Runner
## V16 — Phase 1 (V14 Legacy Pure) + Phase 2B Gate (Per-Question Selectivity)

### Diagnosis from V14/V15:
- **Steering works on hard questions**: +27-35% accuracy gain on baseline-wrong questions
- **Steering breaks easy questions**: -26-28% accuracy loss on baseline-correct questions
- **mag_head saturated**: NLL gradient always pushes magnitude to max, regardless of L1 or init
- **Net effect**: Hard gains cancelled by easy losses → only +1-6pp net

### V16 Strategy — Gate learns WHEN to steer:
1. **Phase 1**: V14 config (legacy_pure, rho_max=0.05) — proven +6.5pp foundation
2. **Phase 2B Gate**: Trained AFTER Phase 1, learns per-question alpha (0→1)
   - Math/hard questions → high alpha (steer strongly)
   - Easy/format questions → low alpha (protect baseline)
   - Uses RLOO reward + contrastive loss + L1 sparsity
3. **Gate epochs = 2**: Quick training, just enough to learn selectivity

### Why this should work:
- Old notebook achieved +9pp (46%→55%) using this exact Phase 1 + Gate approach
- Gate solves the core problem: mag_head can't learn selectivity from NLL, but Gate can learn it from reward signal

## 1. Environment & Hardware Check

In [ ]:
# Set CUDA allocator config BEFORE any torch/transformers import.
# expandable_segments:True lets the allocator satisfy fragmented-memory requests
# without requiring a single contiguous free block — critical for 10-hook training.
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!nvidia-smi
!pip install -q "transformers>=4.36.0" "datasets>=2.16.0" "accelerate>=0.25.0" matplotlib pandas scipy tqdm


## 2. Clone Repository (`development` branch)

In [ ]:
import os
os.environ["PYTHONUNBUFFERED"] = "1"
import sys

REPO_DIR = "/kaggle/working/adaptive-residual-steering"
BRANCH = "development"

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} https://github.com/Abdullah182155/adaptive-residual-steering.git {REPO_DIR}
else:
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull origin {BRANCH}

%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)
print(f"Working directory: {os.getcwd()}")

## 3. Per-Layer Norm Calibration (V5)

In [ ]:
import os
os.chdir(REPO_DIR)
print(f"Current directory: {os.getcwd()}")

# V10: 4 layers matching old notebook [17, 20, 23, 26]
NEW_LAYERS = '17 20 23 26'
print(f"Calibrating norms for 4-layer route: [{NEW_LAYERS}]")
!python scripts/calibrate_norms.py \
    --n_samples 5000 \
    --output norms_calibration.json \
    --candidate_layers {NEW_LAYERS}

print("\n✅ Norm calibration complete:")
!ls -lh norms_calibration.json

## 4. Verify Codebase Integrity & Unit Tests

In [ ]:
import os
import sys
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f"Current directory: {os.getcwd()}")
print(f"Tests directory exists: {os.path.exists('tests/')}")
!pytest tests/ -q --tb=short

## 5. Architecture Configuration

In [ ]:
# v5: rank-64 with causal convolution and per-layer caps
# v4: legacy architecture (default, for NLL training)
STEERNET_ARCH_ARGS = "--steernet_arch v5"
print(f"Architecture: {STEERNET_ARCH_ARGS}")

## 5b. Clear Stale Cache


In [ ]:
# ========== PHASE 1A→1B BUG-6 FIX: Clear Stale Cache ==========
# Cache fingerprint was missing κ and architecture, causing stale output reuse.
# This is now fixed in benchmark.py, but we clear the cache to be safe.
import shutil
import os

cache_dir = "./cache/benchmark_outputs"
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)
    print(f"✅ Cleared stale cache: {cache_dir}")
    os.makedirs(cache_dir, exist_ok=True)
else:
    print(f"✅ Cache directory clean: {cache_dir}")
    os.makedirs(cache_dir, exist_ok=True)

## 6. Hyperparameter Configuration — V16 Phase 1 (V14 Config) + Phase 2B Gate

In [ ]:
# ==============================================================================
# V16: PHASE 1 (V14 LEGACY PURE) + PHASE 2B GATE
# Phase 1 uses V14 proven config (+6.5pp). Gate adds per-question selectivity.
# ==============================================================================
S1_OBJECTIVE = 'nll'
K_SCALE_MODE = 'sqrt'
BATCH_SIZE   = 2
GRAD_ACCUM   = 8
MAX_TOK_LEN  = 512

# ── Architecture: V5 ──
AMPLITUDE_MODE = 'relative'
RHO_MAX        = 0.05          # V14 value — proven +6.5pp
STEER_SCALE    = 0.30          # unused in relative mode
STEER_KAPPA    = 0.15          # soft cap at 15% of calibrated norm
LORA_RANK      = 64
LORA_ALPHA     = 32.0
MAG_HEAD_INIT  = -3.0          # sigmoid(-3)=0.05 — start nearly zero
STEERNET_INIT_GAIN    = 1.0
STEERNET_USE_MAG_HEAD = True
STEERNET_USE_SOFT_CAP = True
STEERNET_USE_CONV     = True
PROMPT_SHIELDING      = False

# ── Loss: LEGACY PURE (V14 proven — uniform NLL on all tokens) ──
LOSS_MODE           = 'legacy_pure'
TYPED_LEARNED_W     = False
LAMBDA_FORMAT       = 0.02
LAMBDA_PROSE        = 0.03
LAMBDA_OPERATOR     = 0.10
LAMBDA_OPERAND      = 0.15
LAMBDA_RESULT       = 0.40
LAMBDA_FINAL        = 0.30
FOCAL_GAMMA         = 2.0
KL_WEIGHT           = 0.0           # No KL in Phase 1 — Gate handles selectivity
KL_ARITH_WEIGHT     = 0.0
MAG_L1_WEIGHT       = 0.10          # Magnitude penalty (still saturates, but Gate will compensate)

# ── Training: 4 epochs, ALL 7473 samples ──
# ╔═══════════════════════════════════════════════════════════════╗
# ║  S1_LR = 1.5e-4  —  DO NOT CHANGE THIS VALUE!               ║
# ║  V14: 1.5e-4 → first +6.5pp gain!                           ║
# ╚═══════════════════════════════════════════════════════════════╝
S1_LR          = 1.5e-4
S1_EPOCHS      = 4
S1_TRAIN_N     = 7473           # ALL available GSM8K training data
S1_VAL_N       = 600
S1_ANSWER_WEIGHT      = 1.0
S1_ANSWER_REGION_FRAC = 0.30
DIVERSIFY_PROMPTS     = True

# ── On-Policy: OFF ──
ON_POLICY = False

# ── Generation / Benchmark ──
BENCH_GEN_TEMP     = 0.0
BENCH_REP_PENALTY  = 1.1
BENCH_NO_REPEAT_NG = 0
BENCH_MAX_TOKENS   = 300

# ── Disabled ──
RUN_CONTRASTIVE = False
RUN_STAR        = False

CALIB_SEL_N  = 100
BENCH_N_TEST = 300
BENCH_SHOTS  = '0 2'

# ── Phase 2B Gate Configuration ──
GATE_EPOCHS              = 2
GATE_LR                  = 1e-4
GATE_FINETUNE_STEERNET   = True
GATE_STEERNET_LR         = 1e-5
GATE_CONTRASTIVE_WEIGHT  = 0.5

print(f'V16 Phase 1+Gate | rank={LORA_RANK}, 4 layers, {S1_EPOCHS} epochs, {S1_TRAIN_N} samples')
print(f'Architecture: V5 gain={STEERNET_INIT_GAIN}, mag_head, conv, soft_cap')
print(f'Loss: LEGACY_PURE (uniform NLL) | rho_max={RHO_MAX} | mag_l1={MAG_L1_WEIGHT}')
print(f'LR: {S1_LR} | On-policy: OFF')
print(f'Gate: {GATE_EPOCHS} epochs, lr={GATE_LR}, finetune_steernet={GATE_FINETUNE_STEERNET}')
print(f'Key change: Phase 1 = V14 proven config, Phase 2B Gate = per-question selectivity')

In [ ]:
# ==============================================================================
# V16: Phase 2B Gate ENABLED — learns per-question steering selectivity
# Router disabled — Gate is the selectivity mechanism
# ==============================================================================
RUN_ROUTER = False
RUN_GATE   = True
BENCH_ARMS = 'baseline,fixed'

print(f'Router: DISABLED | Gate: ENABLED ({GATE_EPOCHS} epochs)')
print(f'Gate learns when to steer (high alpha on hard Qs, low on easy Qs)')
print(f'Benchmark arms: {BENCH_ARMS}')

## 7. Fixed Layer Configuration (Phase 1 V2)


In [ ]:
# ==============================================================================
# V10: 4 layers matching old notebook [17, 20, 23, 26]
# ==============================================================================
# Old notebook used these 4 mid-to-late layers with rank=64 each.
# Fewer layers = more per-layer amplitude, more focused steering.
FIXED_LAYERS = [17, 20, 23, 26]
S_FIXED = ','.join(map(str, FIXED_LAYERS))

print('='*80)
print(f'FIXED LAYER CONFIGURATION (V10 Classic — {len(FIXED_LAYERS)} Layers)')
print('='*80)
print(f'Active layers ({len(FIXED_LAYERS)}): {FIXED_LAYERS}')
print(f'Amplitude mode: {AMPLITUDE_MODE} (steer_scale={STEER_SCALE})')
print('='*80)

## 8. Stage S1: SteerNet Training (Legacy Pure NLL + Magnitude Regularization)

In [ ]:
# Stage S1: SteerNet Training — V5 + Legacy Pure NLL + Magnitude Reg
import subprocess

command = f'''python scripts/run_ars_pipeline.py --stage S1 {STEERNET_ARCH_ARGS} \
    --objective {S1_OBJECTIVE} \
    --s1_epochs {S1_EPOCHS} \
    --s1_lr {S1_LR} \
    --s1_train_n {S1_TRAIN_N} \
    --s1_val_n {S1_VAL_N} \
    --batch_size {BATCH_SIZE} \
    --steer_scale {STEER_SCALE} \
    --s_fixed {S_FIXED} \
    --set steer_kappa={STEER_KAPPA} \
    --set amplitude_mode={AMPLITUDE_MODE} \
    --set rho_max={RHO_MAX} \
    --set k_scale_mode={K_SCALE_MODE} \
    --set grad_accum={GRAD_ACCUM} \
    --set max_tok_len={MAX_TOK_LEN} \
    --set loss_mode={LOSS_MODE} \
    --set lora_rank={LORA_RANK} \
    --set lora_alpha={LORA_ALPHA} \
    --set mag_head_init_bias={MAG_HEAD_INIT} \
    --set diversify_prompts={str(DIVERSIFY_PROMPTS).lower()} \
    --set on_policy=false \
    --set s1_answer_weight={S1_ANSWER_WEIGHT} \
    --set s1_answer_region_frac={S1_ANSWER_REGION_FRAC} \
    --set steernet_init_gain={STEERNET_INIT_GAIN} \
    --set steernet_use_mag_head={str(STEERNET_USE_MAG_HEAD).lower()} \
    --set steernet_use_soft_cap={str(STEERNET_USE_SOFT_CAP).lower()} \
    --set steernet_use_conv={str(STEERNET_USE_CONV).lower()} \
    --set prompt_shielding={str(PROMPT_SHIELDING).lower()} \
    --set mag_l1_weight={MAG_L1_WEIGHT} \
    --set kl_weight={KL_WEIGHT} \
    --set generation_max_new_tokens={BENCH_MAX_TOKENS} \
    --set generation_repetition_penalty={BENCH_REP_PENALTY} \
    --set generation_no_repeat_ngram={BENCH_NO_REPEAT_NG} \
    --set candidate_layers={S_FIXED} \
    --set fixed_layers={S_FIXED}'''

process = subprocess.Popen(
    command, shell=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
for line in process.stdout:
    print(line, end="")
if process.wait() != 0:
    raise RuntimeError(
        f"S1 pipeline failed with exit code {process.returncode}; "
        "full runner output is shown above."
    )

import os, glob
ckpts = sorted(glob.glob("checkpoints/steernet_epoch*.pt"))
print(f'\nPer-epoch checkpoints saved: {[os.path.basename(c) for c in ckpts]}')
print('\n' + '='*80)
print('S1 Training Complete (V16 — legacy_pure, rho_max=0.05)')
print('='*80)

## 8b. Stage S2: Phase 2 Router Training (Per-Question Layer Selection)

In [ ]:
if RUN_ROUTER:
    import subprocess

    command = f'''python scripts/run_ars_pipeline.py --stage S2 {STEERNET_ARCH_ARGS} \
        --s1_train_n {S1_TRAIN_N} \
        --s1_val_n  {S1_VAL_N} \
        --s_fixed   {S_FIXED} \
        --steer_scale {STEER_SCALE} \
        --set steer_kappa={STEER_KAPPA} \
        --set k_scale_mode={K_SCALE_MODE} \
        --set lora_rank={LORA_RANK} \
        --set lora_alpha={LORA_ALPHA} \
        --set phase2_epochs={PHASE2_EPOCHS} \
        --set phase2_lr={PHASE2_LR} \
        --set router_max_k={ROUTER_MAX_K} \
        --set router_min_k={ROUTER_MIN_K} \
        --set router_warmup_epochs={ROUTER_WARMUP} \
        --set router_beta_entropy_subset={ROUTER_BETA_ENTROPY_SUBSET}'''

    process = subprocess.Popen(
        command, shell=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )
    for line in process.stdout:
        print(line, end="")
    if process.wait() != 0:
        print(f"[warning] S2 pipeline exited with code {process.returncode}")

    import torch, json
    ckpt_path = "checkpoints/router.pt"
    if os.path.exists(ckpt_path):
        print('\n' + '='*80)
        print('✅ S2 Router Training Complete')
        print('='*80)
        ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        print(f"Router checkpoint keys: {list(ckpt.keys())}")
        print(f"fixed_layers: {ckpt.get('fixed_layers')}")
        print(f"router_min_k: {ckpt.get('router_min_k')}  router_max_k: {ckpt.get('router_max_k')}")
else:
    print("Router training skipped (RUN_ROUTER=False)")

## 8c. Stage S2B: Phase 2B Gate Training (Per-Token Usefulness Gate)

**Key component from the old notebook:** The gate learned which tokens benefit from steering:
- Math/operator tokens → high alpha (~0.85)
- Format tokens → low alpha (~0.15)  
- Prose/reasoning → medium alpha (~0.65)

This per-token selectivity was responsible for +9pp in the old notebook (46% → 55%).

In [ ]:
if RUN_GATE:
    import subprocess

    command = f'''python scripts/run_ars_pipeline.py --stage S2B {STEERNET_ARCH_ARGS} \
        --s1_train_n {S1_TRAIN_N} \
        --s1_val_n  {S1_VAL_N} \
        --s_fixed   {S_FIXED} \
        --steer_scale {STEER_SCALE} \
        --batch_size {BATCH_SIZE} \
        --set steer_kappa={STEER_KAPPA} \
        --set amplitude_mode={AMPLITUDE_MODE} \
        --set rho_max={RHO_MAX} \
        --set k_scale_mode={K_SCALE_MODE} \
        --set lora_rank={LORA_RANK} \
        --set lora_alpha={LORA_ALPHA} \
        --set mag_head_init_bias={MAG_HEAD_INIT} \
        --set max_tok_len={MAX_TOK_LEN} \
        --set grad_accum={GRAD_ACCUM} \
        --set loss_mode={LOSS_MODE} \
        --set prompt_shielding={str(PROMPT_SHIELDING).lower()} \
        --set use_gate=true \
        --set gate_epochs={GATE_EPOCHS} \
        --set gate_lr={GATE_LR} \
        --set gate_finetune_steernet={str(GATE_FINETUNE_STEERNET).lower()} \
        --set gate_steernet_lr={GATE_STEERNET_LR} \
        --set gate_contrastive_weight={GATE_CONTRASTIVE_WEIGHT} \
        --set candidate_layers={S_FIXED} \
        --set fixed_layers={S_FIXED}'''

    process = subprocess.Popen(
        command, shell=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )
    for line in process.stdout:
        print(line, end="")
    if process.wait() != 0:
        print(f"[warning] S2B pipeline exited with code {process.returncode}")

    # Verify gate checkpoint
    import torch
    gate_path = "checkpoints/gate.pt"
    sn_gated_path = "checkpoints/steernet_gated.pt"
    if os.path.exists(gate_path):
        print('\n' + '='*80)
        print('S2B Gate Training Complete')
        print('='*80)
        ckpt = torch.load(gate_path, map_location="cpu", weights_only=False)
        print(f"Gate checkpoint: best_val_nll={ckpt.get('best_val_nll', 'N/A'):.4f}")
        print(f"Fixed layers: {ckpt.get('fixed_layers')}")
        print(f"Gate epochs: {ckpt.get('gate_epochs')}")
    if os.path.exists(sn_gated_path):
        print(f"Gated SteerNet checkpoint: {sn_gated_path}")
else:
    print("Gate training skipped (RUN_GATE=False)")

## 8d. Stage S1C: Contrastive Residual Training (Reasoning Direction)

**What it does:**
1. Generates greedy completions from the steered model for all training questions
2. Filters to questions where the model is WRONG (hard-example curriculum)
3. Computes the direction vector between correct and incorrect reasoning hidden states
4. Trains SteerNet delta to align with that direction (cosine similarity loss)

The contrastive loss teaches SteerNet WHERE to push in activation space for correct reasoning,
instead of just learning to mimic gold token sequences (which only learns format).

In [ ]:
if RUN_CONTRASTIVE:
    import subprocess

    command = f'''python scripts/run_ars_pipeline.py --stage S1C {STEERNET_ARCH_ARGS} \
        --s1_train_n {S1_TRAIN_N} \
        --s1_val_n {S1_VAL_N} \
        --batch_size {BATCH_SIZE} \
        --s_fixed {S_FIXED} \
        --steer_scale {STEER_SCALE} \
        --set steer_kappa={STEER_KAPPA} \
        --set rho_max={RHO_MAX} \
        --set k_scale_mode={K_SCALE_MODE} \
        --set lora_rank={LORA_RANK} \
        --set lora_alpha={LORA_ALPHA} \
        --set kl_weight={KL_WEIGHT} \
        --set loss_mode={LOSS_MODE} \
        --set grad_accum={GRAD_ACCUM} \
        --set max_tok_len={MAX_TOK_LEN} \
        --set contrastive_phase=true \
        --set contrastive_epochs={CONTRASTIVE_EPOCHS} \
        --set contrastive_lr={CONTRASTIVE_LR} \
        --set contrastive_weight={CONTRASTIVE_WEIGHT} \
        --set contrastive_nll_weight={CONTRASTIVE_NLL_W} \
        --set contrastive_frac={CONTRASTIVE_FRAC} \
        --set candidate_layers={S_FIXED} \
        --set fixed_layers={S_FIXED}'''

    process = subprocess.Popen(
        command, shell=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )
    for line in process.stdout:
        print(line, end="")
    if process.wait() != 0:
        raise RuntimeError(
            f"S1C pipeline failed with exit code {process.returncode}; "
            "full runner output is shown above."
        )

    import os, glob
    ckpts = sorted(glob.glob("checkpoints/steernet_s1c_epoch*.pt"))
    print(f'\nS1C checkpoints: {[os.path.basename(c) for c in ckpts]}')
    print('\n' + '='*80)
    print('S1C Contrastive Training Complete')
    print('='*80)
else:
    print("Contrastive training skipped (RUN_CONTRASTIVE=False)")

## 8e. Stage S1R: Self-Taught Reasoning (STaR)

**What it does:**
1. Generates k=8 completions per question at temp=0.7 with steering ON
2. Scores each: keeps only **correct** completions (rejection sampling)
3. For hard questions (all 8 wrong): adds the gold answer as a **hint**, regenerates,
   then trains on the hint-generated solution WITHOUT the hint
4. Trains SteerNet with standard NLL on these self-generated correct solutions

**Why this works:** Training on gold CoT teaches FORMAT (the gold has specific wording).
Training on the model's OWN correct solutions teaches it to AMPLIFY reasoning paths
it already knows how to produce — much easier for SteerNet to learn.

In [ ]:
if RUN_STAR:
    import subprocess

    command = f'''python scripts/run_ars_pipeline.py --stage S1R {STEERNET_ARCH_ARGS} \
        --s1_train_n {S1_TRAIN_N} \
        --s1_val_n {S1_VAL_N} \
        --batch_size {BATCH_SIZE} \
        --s_fixed {S_FIXED} \
        --steer_scale {STEER_SCALE} \
        --set steer_kappa={STEER_KAPPA} \
        --set rho_max={RHO_MAX} \
        --set k_scale_mode={K_SCALE_MODE} \
        --set lora_rank={LORA_RANK} \
        --set lora_alpha={LORA_ALPHA} \
        --set kl_weight={KL_WEIGHT} \
        --set loss_mode={LOSS_MODE} \
        --set grad_accum={GRAD_ACCUM} \
        --set max_tok_len={MAX_TOK_LEN} \
        --set star_k={STAR_K} \
        --set star_temperature={STAR_TEMP} \
        --set star_epochs={STAR_EPOCHS} \
        --set star_lr={STAR_LR} \
        --set star_rounds={STAR_ROUNDS} \
        --set star_hint_k={STAR_HINT_K} \
        --set star_gen_batch={STAR_GEN_BATCH} \
        --set candidate_layers={S_FIXED} \
        --set fixed_layers={S_FIXED}'''

    process = subprocess.Popen(
        command, shell=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )
    for line in process.stdout:
        print(line, end="")
    if process.wait() != 0:
        raise RuntimeError(
            f"S1R pipeline failed with exit code {process.returncode}; "
            "full runner output is shown above."
        )

    import os, glob
    ckpts = sorted(glob.glob("checkpoints/steernet_s1r_*.pt"))
    print(f'\nS1R checkpoints: {[os.path.basename(c) for c in ckpts]}')
    print('\n' + '='*80)
    print('S1R Self-Taught Reasoning Complete')
    print('='*80)
else:
    print("STaR training skipped (RUN_STAR=False)")

## 9. Stage S5: Deployment Calibration (Contract v3)


In [ ]:
# Stage S5: Calibration — writes deployment_contract.json (version 3)
!python scripts/run_ars_pipeline.py --stage S5 {STEERNET_ARCH_ARGS} \
    --calib_sel_n {CALIB_SEL_N} \
    --contract deployment_contract.json \
    --s_fixed {S_FIXED} \
    --steer_scale {STEER_SCALE} \
    --set steer_kappa={STEER_KAPPA} \
    --set amplitude_mode={AMPLITUDE_MODE} \
    --set rho_max={RHO_MAX} \
    --set k_scale_mode={K_SCALE_MODE} \
    --set lora_rank={LORA_RANK} \
    --set lora_alpha={LORA_ALPHA} \
    --set mag_head_init_bias={MAG_HEAD_INIT} \
    --set steernet_init_gain={STEERNET_INIT_GAIN} \
    --set steernet_use_mag_head={str(STEERNET_USE_MAG_HEAD).lower()} \
    --set steernet_use_soft_cap={str(STEERNET_USE_SOFT_CAP).lower()} \
    --set steernet_use_conv={str(STEERNET_USE_CONV).lower()} \
    --set prompt_shielding={str(PROMPT_SHIELDING).lower()} \
    --set candidate_layers={S_FIXED} \
    --set fixed_layers={S_FIXED}

import json as _json, pathlib as _pl
_c = _json.loads(_pl.Path("deployment_contract.json").read_text())
print(f"\nContract version: {_c.get('version')}  (expected 3)")
assert _c.get("version") == 3, "Expected contract v3!"
print("\n" + "="*80)
print("S5 Calibration Complete (Contract v3 verified)")
print("="*80)

## 10. Benchmark: Baseline vs Fixed-Control (Zero Heuristic Dials)


In [ ]:
# Benchmark: Baseline vs Fixed-control (with Gate if enabled)
_use_gate_flag = 'true' if RUN_GATE else 'false'

!python scripts/run_ars_pipeline.py --stage bench {STEERNET_ARCH_ARGS} \
    --contract deployment_contract.json \
    --n_test {BENCH_N_TEST} \
    --shots {BENCH_SHOTS} \
    --record experiment_record.json \
    --report \
    --arms {BENCH_ARMS} \
    --s_fixed {S_FIXED} \
    --steer_scale {STEER_SCALE} \
    --set steer_kappa={STEER_KAPPA} \
    --set amplitude_mode={AMPLITUDE_MODE} \
    --set rho_max={RHO_MAX} \
    --set lora_rank={LORA_RANK} \
    --set lora_alpha={LORA_ALPHA} \
    --set mag_head_init_bias={MAG_HEAD_INIT} \
    --set steernet_init_gain={STEERNET_INIT_GAIN} \
    --set steernet_use_mag_head={str(STEERNET_USE_MAG_HEAD).lower()} \
    --set steernet_use_soft_cap={str(STEERNET_USE_SOFT_CAP).lower()} \
    --set steernet_use_conv={str(STEERNET_USE_CONV).lower()} \
    --set prompt_shielding={str(PROMPT_SHIELDING).lower()} \
    --set generation_temperature={BENCH_GEN_TEMP} \
    --set generation_repetition_penalty={BENCH_REP_PENALTY} \
    --set generation_no_repeat_ngram={BENCH_NO_REPEAT_NG} \
    --set generation_max_new_tokens={BENCH_MAX_TOKENS} \
    --set use_gate={_use_gate_flag} \
    --set candidate_layers={S_FIXED} \
    --set fixed_layers={S_FIXED}

print('\n' + '='*80)
print('Benchmark Complete (V16 — Phase 1 + Gate)')
print('='*80)

## 11. Deep Diagnostic Report

In [ ]:
import os, json, re, collections, math
from pathlib import Path
import numpy as np
from src.evaluation.chain import audit_chain, gold_step_values, gold_step_recall, classify_outcome

SEP = '=' * 80
THIN = '-' * 60

def _gold_val(gold_text):
    """Extract numeric gold answer — handles both '#### 18' and plain '18' formats."""
    s = str(gold_text).strip()
    if '####' in s:
        s = s.split('####')[-1].strip()
    try:
        return float(s.replace(',', '').replace('$', ''))
    except (ValueError, TypeError):
        return None

# ═══════════════════════════════════════════════════════════════
# LOAD ALL DATA
# ═══════════════════════════════════════════════════════════════
training_log = None
if os.path.exists("report/training_log.json"):
    with open("report/training_log.json", encoding="utf-8") as f:
        training_log = json.load(f)

record = None
if os.path.exists("experiment_record.json"):
    with open("experiment_record.json", encoding="utf-8") as f:
        record = json.load(f)

gen_data = {}
for shot_str in (record or {}).get("shots", {}):
    gen_data[shot_str] = {}
    for arm in ['baseline', 'fixed']:
        path = f"report/generations/gsm8k_{shot_str}shot_{arm}.jsonl"
        if os.path.exists(path):
            with open(path, encoding='utf-8') as f:
                gen_data[shot_str][arm] = [json.loads(line) for line in f if line.strip()]
    paired_path = f"report/generations/gsm8k_{shot_str}shot.jsonl"
    if os.path.exists(paired_path):
        with open(paired_path, encoding='utf-8') as f:
            gen_data[shot_str]['paired'] = [json.loads(line) for line in f if line.strip()]

# ═══════════════════════════════════════════════════════════════
# 1. TRAINING CONVERGENCE
# ═══════════════════════════════════════════════════════════════
print(SEP)
print("  1. TRAINING CONVERGENCE")
print(SEP)
if training_log and training_log.get("epochs"):
    epochs = [ep for ep in training_log["epochs"] if isinstance(ep, dict)]
    print(f"  {'Epoch':<7} {'val_nll':>10} {'crit':>10} {'result_top1':>12}")
    print(f"  {THIN}")
    for i, ep in enumerate(epochs):
        print(f"  {i+1:<7} {ep.get('validation_nll',0):>10.4f} {ep.get('crit',0):>10.4f} {ep.get('result_top1',0):>12.3f}")

    val_nlls = [ep.get('validation_nll', 0) for ep in epochs]
    crits = [ep.get('crit', 0) for ep in epochs]
    if len(val_nlls) >= 2:
        total_drop = val_nlls[0] - val_nlls[-1]
        last_drop = val_nlls[-2] - val_nlls[-1]
        crit_drop = crits[0] - crits[-1]
        print(f"\n  val_nll: {val_nlls[0]:.4f} -> {val_nlls[-1]:.4f} (drop={total_drop:.4f})")
        print(f"  crit:    {crits[0]:.4f} -> {crits[-1]:.4f} (drop={crit_drop:.4f})")
        if val_nlls[-1] > val_nlls[-2]:
            print("  VERDICT: DIVERGING — val_nll increased in last epoch!")
        elif last_drop < 0.002:
            print("  VERDICT: CONVERGED (plateau reached)")
        elif last_drop > 0.01:
            print("  VERDICT: STILL IMPROVING — consider more epochs")
        else:
            print("  VERDICT: CONVERGING (diminishing returns)")
else:
    print("  No training log found.")

# ═══════════════════════════════════════════════════════════════
# 2. ACCURACY TABLE
# ═══════════════════════════════════════════════════════════════
print(f"\n{SEP}")
print("  2. ACCURACY RESULTS")
print(SEP)
if record:
    shots_data = record.get("shots", {})
    for shot_str, shot_data in shots_data.items():
        b = shot_data.get("baseline", {})
        s = shot_data.get("fixed", {})
        paired = shot_data.get("paired", {})

        print(f"\n  [{shot_str}-shot] (n={b.get('n', '?')})")
        print(f"  {'':>18} {'Baseline':>10} {'Fixed':>10} {'Delta':>10}")
        print(f"  {THIN}")

        b_strict = b.get('strict_accuracy', 0)
        s_strict = s.get('strict_accuracy', 0)
        b_len = b.get('lenient_accuracy', 0)
        s_len = s.get('lenient_accuracy', 0)
        b_fmt = b.get('format_rate', 0) * 100
        s_fmt = s.get('format_rate', 0) * 100

        print(f"  {'Strict accuracy':<18} {b_strict:>9.1f}% {s_strict:>9.1f}% {s_strict-b_strict:>+9.1f}pp")
        print(f"  {'Lenient accuracy':<18} {b_len:>9.1f}% {s_len:>9.1f}% {s_len-b_len:>+9.1f}pp")
        print(f"  {'Format rate':<18} {b_fmt:>9.1f}% {s_fmt:>9.1f}% {s_fmt-b_fmt:>+9.1f}pp")

        w = paired.get('fixed_wins', 0)
        l = paired.get('fixed_losses', 0)
        t = paired.get('ties', 0)
        ci = paired.get('ci95_pp', [0, 0])
        p = paired.get('mcnemar_p_value', 1.0)
        sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
        print(f"\n  Paired: {w}W / {l}L / {t}T  |  delta={paired.get('delta_pp',0):+.1f}pp  CI=[{ci[0]:+.1f},{ci[1]:+.1f}]  p={p:.4f} ({sig})")

        guard = s.get('guard_trip_rate', 0)
        ident = paired.get('identical_text_rate', 0)
        print(f"  Guard trips: {guard:.3f}  |  Identical text: {ident:.1%}")

        if s_strict - b_strict > 5 and s_len - b_len < -2:
            print(f"  >> FORMAT-ONLY: strict up but lenient DOWN — learned format, hurt reasoning")
        elif s_len - b_len > 3:
            print(f"  >> REASONING IMPROVED: lenient accuracy gained")
        elif s_len - b_len < -3:
            print(f"  >> REASONING DEGRADED: lenient accuracy dropped")
        elif ident > 0.9:
            print(f"  >> NULL INTERVENTION: >90% identical text — steering has no effect")

# ═══════════════════════════════════════════════════════════════
# 3. STEERING MAGNITUDE (rel_max)
# ═══════════════════════════════════════════════════════════════
print(f"\n{SEP}")
print("  3. STEERING MAGNITUDE (rel_max = ||delta|| / ||hidden||)")
print(SEP)
for shot_str in gen_data:
    fixed = gen_data[shot_str].get('fixed', [])
    baseline = gen_data[shot_str].get('baseline', [])
    if not fixed:
        continue
    rel_maxes = np.array([r.get('rel_max', 0) for r in fixed])
    print(f"\n  {shot_str}-shot fixed ({len(rel_maxes)} questions):")
    print(f"    mean={rel_maxes.mean():.4f}  median={np.median(rel_maxes):.4f}  "
          f"p95={np.percentile(rel_maxes,95):.4f}  max={rel_maxes.max():.4f}")

    std = rel_maxes.std()
    print(f"    std={std:.4f}  min={rel_maxes.min():.4f}  range={rel_maxes.max()-rel_maxes.min():.4f}")

    if rel_maxes.max() > 0.50:
        print(f"    CRITICAL: deltas are {rel_maxes.max():.0%} of hidden norm — overriding model!")
    elif rel_maxes.max() > 0.15:
        print(f"    HIGH: deltas up to {rel_maxes.max():.0%} of hidden norm — may disrupt reasoning")
    elif rel_maxes.max() < 0.005:
        print(f"    TOO WEAK: max rel_max={rel_maxes.max():.4f} — steering negligible")
    else:
        print(f"    OK: deltas are {rel_maxes.max():.1%} of hidden norm — gentle nudge")

    if std < 0.002:
        print(f"    WARNING: std={std:.4f} — mag_head likely SATURATED (same magnitude for all questions)")
    elif std > 0.01:
        print(f"    GOOD: std={std:.4f} — per-question magnitude variation detected")

    # Distribution buckets
    buckets = [(0, 0.005), (0.005, 0.01), (0.01, 0.02), (0.02, 0.03), (0.03, 0.05), (0.05, 0.10), (0.10, 0.50), (0.50, 2.0)]
    for lo, hi in buckets:
        count = ((rel_maxes >= lo) & (rel_maxes < hi)).sum()
        if count > 0:
            print(f"    [{lo:.3f}, {hi:.3f}): {count} questions ({count/len(rel_maxes):.0%})")

# ═══════════════════════════════════════════════════════════════
# 4. REASONING QUALITY (EAA, CAR, GSR)
# ═══════════════════════════════════════════════════════════════
print(f"\n{SEP}")
print("  4. REASONING QUALITY")
print(SEP)
print("  EAA = Exact Arithmetic Accuracy (% of equations computed correctly)")
print("  CAR = Clean Arithmetic Rate (% of responses with ALL equations correct)")
print("  GSR = Gold Step Recall (% of gold intermediate values found in output)")

for shot_str in gen_data:
    paired = gen_data[shot_str].get('paired', [])
    if not paired:
        continue

    metrics = {'b_eaa': [], 'f_eaa': [], 'b_car': [], 'f_car': [],
               'b_gsr': [], 'f_gsr': [], 'b_neq': [], 'f_neq': []}

    for row in paired:
        b_text = row.get('baseline', {}).get('text', '')
        f_text = row.get('fixed', {}).get('text', '')
        question = row.get('question', '')
        gold_text = row.get('gold', '')

        b_audit = audit_chain(b_text, question)
        f_audit = audit_chain(f_text, question)

        metrics['b_eaa'].append(b_audit['eaa'])
        metrics['f_eaa'].append(f_audit['eaa'])
        metrics['b_car'].append(b_audit['car'])
        metrics['f_car'].append(f_audit['car'])
        metrics['b_neq'].append(b_audit['total_equations'])
        metrics['f_neq'].append(f_audit['total_equations'])

        gv = gold_step_values(gold_text)
        metrics['b_gsr'].append(gold_step_recall(b_text, gv))
        metrics['f_gsr'].append(gold_step_recall(f_text, gv))

    print(f"\n  {shot_str}-shot ({len(paired)} questions):")
    print(f"  {'Metric':<28} {'Baseline':>10} {'Fixed':>10} {'Delta':>10}")
    print(f"  {THIN}")
    for label, bk, fk in [
        ('EAA (equation accuracy)', 'b_eaa', 'f_eaa'),
        ('CAR (clean chain rate)', 'b_car', 'f_car'),
        ('GSR (gold step recall)', 'b_gsr', 'f_gsr'),
        ('Avg equations/response', 'b_neq', 'f_neq'),
    ]:
        bv = np.mean(metrics[bk])
        fv = np.mean(metrics[fk])
        d = fv - bv
        if 'neq' in bk:
            print(f"  {label:<28} {bv:>10.1f} {fv:>10.1f} {d:>+10.1f}")
        else:
            print(f"  {label:<28} {bv:>9.1%} {fv:>9.1%} {d:>+9.1%}")

    eaa_d = np.mean(metrics['f_eaa']) - np.mean(metrics['b_eaa'])
    car_d = np.mean(metrics['f_car']) - np.mean(metrics['b_car'])
    gsr_d = np.mean(metrics['f_gsr']) - np.mean(metrics['b_gsr'])

    if eaa_d < -0.03:
        print(f"  >> ARITHMETIC DEGRADED: model makes more calculation errors ({eaa_d:+.1%})")
    if car_d < -0.03:
        print(f"  >> CHAIN CORRUPTED: fewer fully-correct reasoning chains ({car_d:+.1%})")
    if gsr_d < -0.05:
        print(f"  >> REASONING DRIFT: missing gold intermediate values ({gsr_d:+.1%})")
    if eaa_d >= 0 and car_d >= 0 and gsr_d >= 0:
        print(f"  >> Reasoning quality maintained or improved")

# ═══════════════════════════════════════════════════════════════
# 5. ERROR TAXONOMY (per arm)
# ═══════════════════════════════════════════════════════════════
print(f"\n{SEP}")
print("  5. ERROR TAXONOMY")
print(SEP)
print("  Categories: CORRECT | PLAN_ERROR (wrong approach) | ARITHMETIC (bad calc)")
print("  UNGROUNDED (hallucinated number) | EXTRACTION (right intermediate, wrong final)")
print("  NO_ANSWER | LOOP_OR_RUNON | TRUNC_NUMBER")

for shot_str in gen_data:
    paired = gen_data[shot_str].get('paired', [])
    if not paired:
        continue

    b_tax = collections.Counter()
    f_tax = collections.Counter()

    for row in paired:
        b_res = row.get('baseline', {})
        f_res = row.get('fixed', {})
        question = row.get('question', '')
        gv = _gold_val(row.get('gold', ''))
        if gv is None:
            continue

        b_cls = classify_outcome(b_res, gv, question=question)
        f_cls = classify_outcome(f_res, gv, question=question)
        b_tax[b_cls] += 1
        f_tax[f_cls] += 1

    total = sum(b_tax.values())
    print(f"\n  {shot_str}-shot ({total} questions):")
    print(f"  {'Category':<22} {'Baseline':>10} {'Fixed':>10} {'Delta':>10} {'Direction'}")
    print(f"  {THIN}")
    all_cats = sorted(set(list(b_tax.keys()) + list(f_tax.keys())),
                      key=lambda x: -(f_tax[x] + b_tax[x]))
    for cat in all_cats:
        bv = b_tax[cat]
        fv = f_tax[cat]
        d = fv - bv
        arrow = 'BETTER' if (cat == 'CORRECT' and d > 0) or (cat != 'CORRECT' and d < 0) else (
                'WORSE' if (cat == 'CORRECT' and d < 0) or (cat != 'CORRECT' and d > 0) else '=')
        print(f"  {cat:<22} {bv:>10} {fv:>10} {d:>+10} {arrow}")

# ═══════════════════════════════════════════════════════════════
# 6. FLIP ANALYSIS (Win/Loss examples)
# ═══════════════════════════════════════════════════════════════
print(f"\n{SEP}")
print("  6. FLIP ANALYSIS (questions that changed outcome)")
print(SEP)

for shot_str in gen_data:
    paired = gen_data[shot_str].get('paired', [])
    if not paired:
        continue

    wins, losses = [], []
    for row in paired:
        b_ok = row.get('baseline', {}).get('lenient', False)
        f_ok = row.get('fixed', {}).get('lenient', False)
        if f_ok and not b_ok:
            wins.append(row)
        elif b_ok and not f_ok:
            losses.append(row)

    print(f"\n  {shot_str}-shot: {len(wins)} WINS, {len(losses)} LOSSES")

    if wins:
        print(f"\n  -- WINS (baseline wrong -> fixed right) --")
        win_reasons = collections.Counter()
        for w in wins:
            gv = _gold_val(w.get('gold', ''))
            if gv is None:
                continue
            cls = classify_outcome(w['baseline'], gv, question=w.get('question', ''))
            win_reasons[f"fixed {cls}"] += 1
        for reason, count in win_reasons.most_common(5):
            print(f"    {count:>3}x baseline was {reason}")
        for w in wins[:3]:
            ga = _gold_val(w.get('gold', ''))
            bp = w['baseline'].get('pred_lenient', '?')
            fp = w['fixed'].get('pred_lenient', '?')
            print(f"\n    Q: {w.get('question','')[:90]}...")
            print(f"    Gold={ga}  Baseline={bp}  Fixed={fp}")

    if losses:
        print(f"\n  -- LOSSES (baseline right -> fixed wrong) --")
        loss_reasons = collections.Counter()
        for lo in losses:
            gv = _gold_val(lo.get('gold', ''))
            if gv is None:
                continue
            cls = classify_outcome(lo['fixed'], gv, question=lo.get('question', ''))
            loss_reasons[cls] += 1
        for reason, count in loss_reasons.most_common(5):
            print(f"    {count:>3}x fixed error: {reason}")
        for lo in losses[:3]:
            ga = _gold_val(lo.get('gold', ''))
            bp = lo['baseline'].get('pred_lenient', '?')
            fp = lo['fixed'].get('pred_lenient', '?')
            print(f"\n    Q: {lo.get('question','')[:90]}...")
            print(f"    Gold={ga}  Baseline={bp}  Fixed={fp}")
            ft = lo['fixed'].get('text', '')
            if ft:
                lines = ft.strip().split('\n')
                print(f"    Fixed output ({len(lines)} lines):")
                for line in lines[:5]:
                    print(f"      {line}")
                if len(lines) > 5:
                    print(f"      ... ({len(lines)-5} more lines)")

# ═══════════════════════════════════════════════════════════════
# 7. GENERATION STATS
# ═══════════════════════════════════════════════════════════════
print(f"\n{SEP}")
print("  7. GENERATION STATS")
print(SEP)
for shot_str in gen_data:
    for arm in ['baseline', 'fixed']:
        data = gen_data[shot_str].get(arm, [])
        if not data:
            continue
        stops = collections.Counter(r.get('stop_reason', '?') for r in data)
        tokens = [r.get('n_new_tokens', 0) for r in data]
        print(f"  {shot_str}-shot {arm}: tokens mean={np.mean(tokens):.0f} med={np.median(tokens):.0f} max={max(tokens)}  |  stops: {dict(stops)}")

# ═══════════════════════════════════════════════════════════════
# 8. DIFFICULTY ANALYSIS (easy vs hard questions)
# ═══════════════════════════════════════════════════════════════
print(f"\n{SEP}")
print("  8. DIFFICULTY STRATIFICATION")
print(SEP)
print("  Easy = baseline gets right, Hard = baseline gets wrong")

for shot_str in gen_data:
    paired = gen_data[shot_str].get('paired', [])
    if not paired:
        continue

    easy_b, easy_f, hard_b, hard_f = 0, 0, 0, 0
    n_easy, n_hard = 0, 0
    for row in paired:
        b_ok = row.get('baseline', {}).get('lenient', False)
        f_ok = row.get('fixed', {}).get('lenient', False)
        if b_ok:
            n_easy += 1
            easy_b += 1
            easy_f += int(f_ok)
        else:
            n_hard += 1
            hard_f += int(f_ok)

    if n_easy > 0 and n_hard > 0:
        print(f"\n  {shot_str}-shot:")
        print(f"    Easy ({n_easy}): baseline={easy_b/n_easy:.0%} -> fixed={easy_f/n_easy:.0%}  "
              f"(retention={easy_f/n_easy:.0%}, lost={n_easy-easy_f})")
        print(f"    Hard ({n_hard}): baseline=0% -> fixed={hard_f/n_hard:.0%}  "
              f"(gained={hard_f})")
        net = hard_f - (n_easy - easy_f)
        print(f"    Net: +{hard_f} gained on hard, -{n_easy-easy_f} lost on easy = {net:+d} net flips")
        if n_easy - easy_f > hard_f:
            print(f"    >> PROBLEM: losing more easy questions than gaining hard ones!")
        elif hard_f > 0 and n_easy == easy_f:
            print(f"    >> EXCELLENT: gaining on hard without losing any easy ones!")

# ═══════════════════════════════════════════════════════════════
# 9. AUTO-DIAGNOSIS SUMMARY
# ═══════════════════════════════════════════════════════════════
print(f"\n{SEP}")
print("  9. AUTO-DIAGNOSIS")
print(SEP)
signals = []
if record:
    for shot_str, shot_data in record.get("shots", {}).items():
        b = shot_data.get("baseline", {})
        s = shot_data.get("fixed", {})
        paired = shot_data.get("paired", {})

        len_d = s.get('lenient_accuracy', 0) - b.get('lenient_accuracy', 0)
        str_d = s.get('strict_accuracy', 0) - b.get('strict_accuracy', 0)
        fmt_d = (s.get('format_rate', 0) - b.get('format_rate', 0)) * 100
        ident = paired.get('identical_text_rate', 0)

        fixed_gen = gen_data.get(shot_str, {}).get('fixed', [])
        rel_maxes = np.array([r.get('rel_max', 0) for r in fixed_gen]) if fixed_gen else np.array([0])
        max_rm = rel_maxes.max()
        std_rm = rel_maxes.std()

        if ident > 0.90:
            signals.append(f"  [{shot_str}s] NULL INTERVENTION: {ident:.0%} identical text — steering has NO effect")
            if max_rm < 0.005:
                signals.append(f"         Cause: rel_max={max_rm:.4f} — deltas too small to change output")
        elif str_d > 10 and len_d < -2:
            signals.append(f"  [{shot_str}s] FORMAT-ONLY: strict +{str_d:.0f}pp but lenient {len_d:+.0f}pp")
            signals.append(f"         Cause: learned 'Final answer:' format but corrupted reasoning")
            if max_rm > 0.3:
                signals.append(f"         rel_max={max_rm:.2f} — deltas too large, overriding base model")
        elif len_d < -5:
            signals.append(f"  [{shot_str}s] SIGNIFICANT REGRESSION: lenient {len_d:+.0f}pp")
            if max_rm > 0.3:
                signals.append(f"         Likely cause: over-steering (rel_max={max_rm:.2f})")
            else:
                signals.append(f"         rel_max={max_rm:.3f} — check error taxonomy for root cause")
        elif len_d > 5:
            signals.append(f"  [{shot_str}s] GOOD IMPROVEMENT: lenient {len_d:+.0f}pp (p={paired.get('mcnemar_p_value',1):.4f})")
        elif abs(len_d) <= 2:
            signals.append(f"  [{shot_str}s] NEUTRAL: lenient {len_d:+.1f}pp — within noise")
            if max_rm < 0.01:
                signals.append(f"         Possible cause: under-steering (rel_max={max_rm:.4f})")
            if std_rm < 0.002 and max_rm > 0.01:
                signals.append(f"         Possible cause: mag_head saturated (std={std_rm:.4f}) — no per-question selectivity")
        else:
            signals.append(f"  [{shot_str}s] MILD {'GAIN' if len_d > 0 else 'DROP'}: lenient {len_d:+.1f}pp")

        if std_rm < 0.002 and max_rm > 0.01:
            signals.append(f"  [{shot_str}s] MAG_HEAD SATURATED: std={std_rm:.4f}, all questions get same magnitude")

for s in signals:
    print(s)

if not signals:
    print("  No data available for diagnosis.")

print(f"\n{SEP}")
print("  END OF DIAGNOSTIC REPORT")
print(SEP)

# ═══════════════════════════════════════════════════════════════
# TRAINING CURVES PLOT (saved to report/)
# ═══════════════════════════════════════════════════════════════
if training_log:
    from src.evaluation.diagnostics import plot_training
    plot_training(training_log, "report/training_curves.png")
    print("\nTraining curves saved to report/training_curves.png")

## 12. Save Artifacts


In [ ]:
import shutil, os, json, glob

os.makedirs("/kaggle/working/ars_artifacts", exist_ok=True)

# Core artifacts
for fname in [
    "deployment_contract.json",
    "experiment_record.json",
    "norms_calibration.json",
    "checkpoints/steernet.pt",
    "checkpoints/router.pt",
    "checkpoints/gate.pt",
    "checkpoints/steernet_gated.pt",
]:
    if os.path.exists(fname):
        shutil.copy(fname, "/kaggle/working/ars_artifacts/")
        print(f"Copied: {fname}")

# Per-epoch SteerNet checkpoints (for ablation)
for ckpt in sorted(glob.glob("checkpoints/steernet_epoch*.pt")):
    shutil.copy(ckpt, "/kaggle/working/ars_artifacts/")
    print(f"Copied: {ckpt}")

if os.path.exists("report"):
    shutil.copytree("report", "/kaggle/working/ars_artifacts/report", dirs_exist_ok=True)
    print("Copied: report/")

# Verify contract version in artifact
_c_path = "/kaggle/working/ars_artifacts/deployment_contract.json"
if os.path.exists(_c_path):
    _c = json.loads(open(_c_path).read())
    print(f"\nArtifact contract version: {_c.get('version')}")
    assert _c.get("version") == 3, "Contract must be v3 in artifact!"
    assert "shot_scales" not in _c, "shot_scales must not be in artifact contract!"
    print("✅ Contract v3 artifact verified")

print("\nArtifacts saved to /kaggle/working/ars_artifacts:")
!ls -lh /kaggle/working/ars_artifacts

## 13. Export Benchmark Bundle (ZIP)

In [ ]:
import zipfile, os, shutil
from pathlib import Path

# Files needed to run any benchmark on another machine
BUNDLE_FILES = [
    "deployment_contract.json",   # route + envelope metadata
    "norms_calibration.json",     # per-layer norm caps (required by SteerNet)
    "checkpoints/steernet.pt",    # Phase 1 SteerNet weights
    "checkpoints/router.pt",      # Phase 2 Router weights (if trained)
]

ZIP_PATH = "/kaggle/working/ars_benchmark_bundle.zip"

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for fpath in BUNDLE_FILES:
        if os.path.exists(fpath):
            zf.write(fpath)
            size_mb = os.path.getsize(fpath) / 1e6
            print(f"  + {fpath}  ({size_mb:.1f} MB)")
        else:
            print(f"  - {fpath}  [NOT FOUND — skipped]")

total_mb = os.path.getsize(ZIP_PATH) / 1e6
print(f"\n✅ Bundle saved: {ZIP_PATH}  ({total_mb:.1f} MB)")
print("Download this single file to run benchmarks on any machine.")